# two-optimizers-alternating-step — faded example 3: Fresh noise per critic step

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `two-optimizers-alternating-step`. The last cell reports your progress on the `GAN: Two-optimizers alternating step` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Two-optimizers alternating step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`two-optimizers-alternating-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "two-optimizers-alternating-step"
DD_SUBTOPIC = "GAN: Two-optimizers alternating step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

WGAN's inner critic loop samples a brand-new `z = t.randn(B, z_dim)` on every step rather than reusing one batch. Fresh noise per step keeps the critic from overfitting to a single fake sample and is part of the n_critic recipe.

## Faded exercise 3

### Sample fresh noise inside the critic loop

Implement `critic_loop(G, D, D_opt, x_real, z_dim, n_critic)` returning the list of critic losses. Each inner step must draw its own noise. Complete the line that samples a fresh `(B, z_dim)` noise batch.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

def critic_loop(G, D, D_opt, x_real, z_dim, n_critic):
    B = x_real.shape[0]
    losses = []
    for _ in range(n_critic):
        D_opt.zero_grad()
        z = None  # TODO: fill in this step — read the prompt cell above
        fake = G(z).detach()
        loss_D = (D(fake) - D(x_real)).mean()
        loss_D.backward(); D_opt.step()
        losses.append(loss_D.item())
    return losses

t.manual_seed(0)
G = nn.Linear(4, 5); D = nn.Linear(5, 1)
D_opt = t.optim.SGD(D.parameters(), lr=0.01)
print(critic_loop(G, D, D_opt, t.randn(6, 5), 4, 3))


def _test():
    t.manual_seed(0)
    G = nn.Linear(4, 5); D = nn.Linear(5, 1)
    D_opt = t.optim.SGD(D.parameters(), lr=0.01)
    x_real = t.randn(6, 5)
    losses = critic_loop(G, D, D_opt, x_real, z_dim=4, n_critic=3)
    # exactly n_critic recorded losses
    assert len(losses) == 3, len(losses)
    assert all(isinstance(l, float) for l in losses)
    # the z must be (B, z_dim): if a wrong shape were sampled, G(z) would raise,
    # so reaching here with 3 floats already implies the shape was correct.


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def critic_loop(G, D, D_opt, x_real, z_dim, n_critic):
    B = x_real.shape[0]
    losses = []
    for _ in range(n_critic):
        D_opt.zero_grad()
        z = t.randn(B, z_dim)
        fake = G(z).detach()
        loss_D = (D(fake) - D(x_real)).mean()
        loss_D.backward(); D_opt.step()
        losses.append(loss_D.item())
    return losses

t.manual_seed(0)
G = nn.Linear(4, 5); D = nn.Linear(5, 1)
D_opt = t.optim.SGD(D.parameters(), lr=0.01)
print(critic_loop(G, D, D_opt, t.randn(6, 5), 4, 3))
```
</details>